# 03 May 05 Default Linear Clip + DQN LayerNorm

Keeps the tuned LinearBidder bin clip fixed and tests DQN LayerNorm against the fixed-clip default control.

In [6]:
import sys
import json
import pickle
from dataclasses import replace
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.shared_runner import run_experiment_inprocess


In [7]:
_linear_scr_fpa = Path(REPO_ROOT) / 'example_notebooks' / 'evaluate_baselines' / 'best_params' / 'fpa_baseline_n10_rndm_42' / 'linear_scr_FPA.pkl'
with _linear_scr_fpa.open('rb') as f:
    linear_tuned_params = pickle.load(f)
linear_tuned_params


{'coef': 0.023335213830958296,
 'lower_clip': 9,
 'upper_clip': 1,
 'factor': 3.4224852046754637}

In [8]:
RUN_NAME = 'may05_default_linear_clip_dqn_layer_norm_fixed'
DRLB_PROFILE = 'may05_default_linear_clip_dqn_layer_norm_fixed'
VERBOSE = False


In [9]:
config = build_drlb_config(RUN_NAME, profile=DRLB_PROFILE, split_set='full_train_val_holdout')
config = replace(config, n_trials=10, refit_on='train_plus_val', max_steps=None)

profile_data = get_drlb_profile(DRLB_PROFILE)
base_drlb_params = dict(profile_data['base_drlb_params'])
reference_model_params = dict(profile_data['reference_model_params'])
base_drlb_params['bid_lower_clip'] = linear_tuned_params['lower_clip']
base_drlb_params['bid_upper_clip'] = linear_tuned_params['upper_clip']
base_drlb_params['traffic_path'] = str(REPO_ROOT / 'data' / 'traffic_share.csv')

result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    base_drlb_params=base_drlb_params,
    reference_model_params=reference_model_params,
    state_type=profile_data['state_type'],
    objective=profile_data['objective'],
    search_space_fn=profile_data['search_space_fn'],
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

summary = result['summary']
{
    'run_name': config.run_name,
    'profile': DRLB_PROFILE,
    'linear_tuned_params': linear_tuned_params,
    'base_drlb_params': base_drlb_params,
    'reference_model_params': reference_model_params,
    'best_val_metrics': summary['tuning']['best_val_metrics'],
    'final_holdout_metrics': summary['final_holdout']['metrics'],
    'diagnostics_png': summary['refit']['combined_diagnostics_plot_path'],
}


[I 2026-05-05 11:53:18,100] A new study created in memory with name: no-name-28221125-80a6-4346-bb75-9bbbee38c1a9
[I 2026-05-05 11:58:18,176] Trial 0 finished with value: 2147.862898747218 and parameters: {}. Best is trial 0 with value: 2147.862898747218.
[I 2026-05-05 12:02:39,758] Trial 1 finished with value: 2147.862898747218 and parameters: {}. Best is trial 0 with value: 2147.862898747218.
[I 2026-05-05 12:06:57,763] Trial 2 finished with value: 2147.862898747218 and parameters: {}. Best is trial 0 with value: 2147.862898747218.
[I 2026-05-05 12:11:05,233] Trial 3 finished with value: 2147.862898747218 and parameters: {}. Best is trial 0 with value: 2147.862898747218.
[I 2026-05-05 12:15:16,978] Trial 4 finished with value: 2147.862898747218 and parameters: {}. Best is trial 0 with value: 2147.862898747218.
[I 2026-05-05 12:19:37,370] Trial 5 finished with value: 2147.862898747218 and parameters: {}. Best is trial 0 with value: 2147.862898747218.
[I 2026-05-05 12:23:38,774] Trial 

{'run_name': 'may05_default_linear_clip_dqn_layer_norm_fixed',
 'profile': 'may05_default_linear_clip_dqn_layer_norm_fixed',
 'linear_tuned_params': {'coef': 0.023335213830958296,
  'lower_clip': 9,
  'upper_clip': 1,
  'factor': 3.4224852046754637},
 'base_drlb_params': {'max_bid': 100.0,
  'T': 72,
  'lambda_min': -inf,
  'lambda_max': inf,
  'bids_per_timestep': 1,
  'dqn_soft_update_tau': 0.01,
  'dqn_loss_type': 'smooth_l1',
  'dqn_grad_clip_norm': 5.0,
  'dqn_reward_clip_value': 10.0,
  'init_lambda': 0.0028423174374845716,
  'init_lambda_mode': 'constant',
  'bid_lower_clip': 9,
  'bid_upper_clip': 1,
  'dqn_layer_norm': True,
  'traffic_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/traffic_share.csv'},
 'reference_model_params': {'dqn_gamma': 1.0,
  'dqn_lr': 0.0003,
  'dqn_target_update_interval': 100,
  'reward_net_lr': 0.01,
  'dqn_epsilon_start': 0.95,
  'dqn_epsilon_end': 0.05,
  'dqn_epsilon_anneal': 2e-05},
 'best_val_metrics': {'cpc_relative': 4

In [10]:
pd.DataFrame([
    {'artifact': 'run_summary_json', 'path': str(config.outputs_dir / 'run_summary.json')},
    {'artifact': 'metrics_json', 'path': str(config.outputs_dir / 'metrics.json')},
    {'artifact': 'drlb_diagnostics_png', 'path': str(config.outputs_dir / 'drlb_diagnostics.png')},
    {'artifact': 'best_refit_model', 'path': str(config.best_models_dir / 'best_refit.pt')},
])


,artifact,path
0,run_summary_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
1,metrics_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
2,drlb_diagnostics_png,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
3,best_refit_model,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
